In [1]:
import os
import json
from tqdm import tqdm

def read_json_file(path: str) -> dict | list:
    with open(path, 'r') as f:
        return json.load(f)

def write_json_file(data: dict | list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [2]:
# Install transformers if needed and load Qwen tokenizer
from transformers import AutoTokenizer
from copy import deepcopy

# Load Qwen tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B", trust_remote_code=True)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
generation_tag_data = read_json_file("./data/raw_data/Generation/alpaca_dataset.json")
for d in generation_tag_data:
    d["instruction"] = d["instruction"].replace("<Generate_Response>", "").replace("</Generate_Response>", "").strip()
    d["instruction"] = json.loads(d["instruction"])
    d["output"] = json.loads(d["output"])

print(len(generation_tag_data))
generation_tag_data[0]

1746


{'instruction': {'title': 'راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس\u200cهای ارزش افزوده',
  'language': 'PERSIAN',
  'context': 'IMPORTANT DOCUMENTS:\n[Doc 0]\nQuestion: `تحویل فیزیکی طلا`, Answer: بدلیل شرایط جنگی کشور، تحویل فیزیکی طلا تا اطلاع ثانوی غیرفعال شده است.\n\n\nREGULAR DOCUMENTS:\n[Doc 0]\nراهنمای خرید و فروش طلا در اپلیکیشن بانکت\nتحویل فیزیکی طلا\nعیار طلای تحویل شده و انتخاب آن\n**عیار طلای تحویل شده:**\n\n\n\nاگرچه خرید و فروش طلا در اپلیکیشن بانکت، فقط براساس قیمت طلا با عیار 18 انجام میشود،\nاما هنگام تحویل فیزیکی، امکان انتخاب انواع شمش\u200cها با عیار 18 و 24 وجود دارد.\nدرصورت انتخاب شمش با عیار 24، میزان موجودی طلای کاربر،\nاز عیار 18 به عیار 24 تبدیل شده و متناسب با آن، امکان انتخاب شمش، وجود خواهد داشت.\n\t Reference:\n\t 7_Gold_V_5.docx\n\n[Doc 1]\nراهنمای خرید و فروش طلا در اپلیکیشن بانکت\nتسویه وجه حاصل از فروش طلا\nفرآیند تسویه وجه پس از فروش طلا\n## تسویه وجه طلا – تسویه وجه حاصل از فروش طلا:\n\n\n\nپس از فروش طلا و ارسال درخواست برداشت وجه ا

In [4]:
def format_generation_input(input_data: dict) -> str:
    text = f"""<GenerationResponse>\
<name>{input_data['name']}</name>\
<title>{input_data['title']}</title>\
<context>{input_data['context']}</context>\
<history>{input_data['history']}</history>\
<message>{input_data['message']}</message>\
<language>{input_data['language']}</language>\
</GenerationResponse>"""    
    return text

In [5]:
def convert_generation_to_alpaca_format(raw_data: list) -> list:
    final_data = []
    for item in tqdm(raw_data):
        final_data.append({
            "num_tokens": len(tokenizer.encode(format_generation_input(item["instruction"]))),
            "tag": item["output"]["tag"],
            "instruction": format_generation_input(item["instruction"]),
            "input": "",
            "output": json.dumps(item["output"], ensure_ascii=False)
        })
    return final_data

In [6]:
generation_tag_alpaca_data = convert_generation_to_alpaca_format(generation_tag_data)

100%|██████████| 1746/1746 [00:17<00:00, 102.56it/s]


In [7]:
generation_tag_alpaca_data = sorted(generation_tag_alpaca_data, key=lambda x: x["num_tokens"], reverse=False)

In [8]:
query_refiner_data = read_json_file("./data/raw_data/QueryRefiner/QueryRefiner.json")
for d in query_refiner_data:
    d["input_data"] = json.loads(d["input_data"])
    d["output_data"] = json.loads(d["output_data"])

print(len(query_refiner_data))
query_refiner_data[0]

2581


{'trace_id': '47a28a4cf846642dd73af5d412aa3133',
 'chain_name': 'QueryRefiner',
 'input_data': {'user_input': 'سلام ب نظرت میتونی کمکی بکنی؟\nدرخواست کارت دادم بیشتراز ده روزه هنوز ب دستم نرسیده',
  'history': '',
  'title': 'راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس\u200cهای ارزش افزوده',
  'language': 'PERSIAN'},
 'output_data': {'output': 'سلام. به نظرت میتونی کمکی بکنی؟ من درخواست صدور کارت بانکت دادم و بیش از ده روزه که هنوز به دستم نرسیده.'},
 'metadata': {'completion_tokens': 48,
  'prompt_tokens': 1534,
  'total_cost': 0.00018209999999999998},
 'collection_info_version': 173}

In [9]:
def format_refiner_input(input_data: dict) -> str:
    text = f"""<QueryRefiner>\
<title>{input_data['title']}</title>\
<history>{input_data['history']}</history>\
<user_input>{input_data['user_input']}</user_input>\
<language>{input_data['language']}</language>\
</QueryRefiner>"""    
    return text

In [10]:
def convert_refiner_to_alpaca_format(raw_data: list) -> list:
    final_data = []
    for item in tqdm(raw_data):
        final_data.append({
            "num_tokens": len(tokenizer.encode(format_refiner_input(item["input_data"]))),
            "tag": "QueryRefiner",
            "instruction": format_refiner_input(item["input_data"]),
            "input": "",
            "output": f'{item["output_data"]}'
        })
    return final_data

In [11]:
refiner_alpaca_data = convert_refiner_to_alpaca_format(query_refiner_data)

100%|██████████| 2581/2581 [00:02<00:00, 1209.50it/s]


In [12]:
refiner_alpaca_data = sorted(refiner_alpaca_data, key=lambda x: x["num_tokens"], reverse=False)

In [ ]:
max_num_tokens = 2000  # LLaMA-Factory adds some special tokens, so we set a slightly lower limit.
tmp_mixed_data = []
for d in tqdm(generation_tag_alpaca_data + refiner_alpaca_data):
    if d["num_tokens"] <= max_num_tokens:
        tmp_mixed_data.append(d)

100%|██████████| 4327/4327 [00:00<00:00, 1214694.69it/s]


In [14]:
# Count tokens for each instruction
token_counts = {}
for item in tqdm(tmp_mixed_data):
    token_counts.setdefault(item["tag"], []).append(item["num_tokens"])

import numpy as np

f"Token count statistics for each tag:\n"
for tag, counts in token_counts.items():
    print(f"Tag: {tag}")
    print(f"  Total items: {len(counts)}")
    print(f"  Min tokens: {min(counts)}")
    print(f"  Max tokens: {max(counts)}")
    print(f"  Mean tokens: {np.mean(counts):.2f}")
    print(f"  Median tokens: {np.median(counts):.2f}")
    print(f"  Total tokens: {sum(counts)}")
    print()

100%|██████████| 2817/2817 [00:00<00:00, 866355.36it/s]

Tag: not_enough
  Total items: 25
  Min tokens: 120
  Max tokens: 1995
  Mean tokens: 1749.72
  Median tokens: 1828.00
  Total tokens: 43743

Tag: normal
  Total items: 211
  Min tokens: 1321
  Max tokens: 1999
  Mean tokens: 1828.50
  Median tokens: 1850.00
  Total tokens: 385813

Tag: QueryRefiner
  Total items: 2581
  Min tokens: 57
  Max tokens: 915
  Mean tokens: 217.96
  Median tokens: 140.00
  Total tokens: 562553



In [15]:
final_mixed_data = []

counter = {
    "normal": 0,
    "not_enough": 0,
    "QueryRefiner": 0
}
max_items = {
    "normal": None,
    "not_enough": None,
    "QueryRefiner": 100
}
for d in tmp_mixed_data:
    if (max_items[d["tag"]] is None) or (counter[d["tag"]] < max_items[d["tag"]]):
        final_mixed_data.append(d)
        counter[d["tag"]] += 1


In [16]:
sum(counter.values()), counter

(336, {'normal': 211, 'not_enough': 25, 'QueryRefiner': 100})

In [17]:
import random
random.shuffle(final_mixed_data)

In [18]:
write_json_file(final_mixed_data, "./data/processed_data/Mix_RefinerGeneration_Alpaca336.json")